# SCD Type 1 — Overwrite (No History)

**SCD Type 1** is the simplest strategy: when a dimension attribute changes the existing row is simply **overwritten**.  
No history is kept — only the current state of each record is stored.

### When to use Type 1
- The change is a **correction** (a typo, a data-quality fix) and the old value is meaningless.
- History of the attribute is not required by the business.
- Storage efficiency is a concern.

### Table structure
| Column | Purpose |
|---|---|
| `id` | Natural/business key |
| `name` | The tracked attribute (overwritten on change) |
| `last_updated` | Timestamp of the most recent upsert |

### Techniques covered
1. Create the Type 1 Delta table
2. Initial load from source CSV
3. MERGE-based upsert — insert new rows, overwrite changed rows
4. Verify the overwrite (old value gone)
5. Insert-only MERGE — insert new source rows, ignore existing
6. Delete-and-reload pattern (full refresh)

## Setup

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from delta.tables import DeltaTable

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

## 1. Read the source CSV

In [ ]:
CSV_PATH = "/Volumes/population_metrics/landing/datasets/countries_dataset/countries_dataset/csv_data/country_regions/country_regions.csv"

source_schema = StructType([
    StructField("id",   IntegerType(), False),
    StructField("name", StringType(),  True)
])

source_df = spark.read.format("csv").schema(source_schema).option("header", True).load(CSV_PATH)
source_df.show()

## 2. Create the SCD Type 1 Delta table

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS population_metrics.default.scd1_country_regions (
        id           INT     NOT NULL,
        name         STRING,
        last_updated TIMESTAMP
    )
    USING DELTA
    COMMENT 'SCD Type 1 dimension table for world regions — overwrites on change'
""")
print("Table ready.")

## 3. Initial load

Insert all source rows. The guard ensures re-running the notebook is safe.

In [ ]:
row_count = spark.table("population_metrics.default.scd1_country_regions").count()

if row_count == 0:
    initial_df = source_df.withColumn("last_updated", F.current_timestamp())
    initial_df.write.format("delta").mode("append").saveAsTable("population_metrics.default.scd1_country_regions")
    print(f"Seeded {initial_df.count()} rows.")
else:
    print(f"Table already has {row_count} rows — skipping initial load.")

spark.table("population_metrics.default.scd1_country_regions").orderBy("id").show(truncate=False)

## 4. MERGE-based upsert — the core Type 1 technique

When the source delivers updated data we use MERGE to:
- **UPDATE** existing rows whose `name` has changed (overwrite — history is lost)
- **INSERT** rows whose `id` does not yet exist in the target

Here we simulate: America renamed → *North & South America*, and Antarctica (id=60) added.

In [ ]:
from pyspark.sql import Row

updated_rows = [
    Row(id=10, name="North & South America"),  # changed
    Row(id=20, name="Europe"),
    Row(id=30, name="Asia"),
    Row(id=40, name="Oceania"),
    Row(id=50, name="Africa"),
    Row(id=60, name="Antarctica"),              # new
]
updated_df = spark.createDataFrame(updated_rows, schema=source_schema) \
                  .withColumn("last_updated", F.current_timestamp())

scd1 = DeltaTable.forName(spark, "population_metrics.default.scd1_country_regions")

scd1.alias("target").merge(
    updated_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(
    condition="target.name <> source.name",   # only update if something actually changed
    set={
        "name":         "source.name",
        "last_updated": "source.last_updated"
    }
).whenNotMatchedInsertAll() \
 .execute()

print("Merge complete.")
spark.table("population_metrics.default.scd1_country_regions").orderBy("id").show(truncate=False)

## 5. Verify — old value is gone

Unlike Type 2, the original "America" row no longer exists.  
Only the new name is stored. This is the defining trade-off of Type 1.

In [ ]:
print("=== Current state of the table ===")
spark.table("population_metrics.default.scd1_country_regions").orderBy("id").show(truncate=False)

print("=== Searching for the old name 'America' ===")
result = spark.sql("""
    SELECT * FROM population_metrics.default.scd1_country_regions WHERE name = 'America'
""")
if result.count() == 0:
    print("'America' not found — the old value was overwritten as expected.")
else:
    result.show(truncate=False)

## 6. Insert-only MERGE (ignore existing rows)

Sometimes you only want to add **new** source rows and leave existing records untouched.  
Omitting `whenMatchedUpdate` achieves this cleanly.

In [ ]:
new_rows = [
    Row(id=70, name="Arctic"),   # new only
    Row(id=10, name="Should be ignored"),  # existing id — will NOT be inserted or updated
]
new_df = spark.createDataFrame(new_rows, schema=source_schema) \
              .withColumn("last_updated", F.current_timestamp())

scd1.alias("target").merge(
    new_df.alias("source"),
    "target.id = source.id"
).whenNotMatchedInsertAll() \
 .execute()

print("Insert-only merge complete.")
spark.table("population_metrics.default.scd1_country_regions").orderBy("id").show(truncate=False)

## 7. Delete-and-reload (full refresh)

The bluntest Type 1 strategy: truncate the target and reload from source.  
Simple and reliable when the source is a complete snapshot and performance is not a concern.

Delta Lake makes this safe — `TRUNCATE` still creates a new version in the transaction log.

In [ ]:
# Full snapshot that represents the authoritative source state
full_snapshot = [
    Row(id=10, name="North & South America"),
    Row(id=20, name="Europe"),
    Row(id=30, name="Asia"),
    Row(id=40, name="Oceania"),
    Row(id=50, name="Africa"),
    Row(id=60, name="Antarctica"),
]
snapshot_df = spark.createDataFrame(full_snapshot, schema=source_schema) \
                   .withColumn("last_updated", F.current_timestamp())

# Overwrite the entire table — Delta creates a new version, history still accessible via time travel
snapshot_df.write.format("delta").mode("overwrite").option("overwriteSchema", "false") \
           .saveAsTable("population_metrics.default.scd1_country_regions")

print("Full refresh complete.")
spark.table("population_metrics.default.scd1_country_regions").orderBy("id").show(truncate=False)

## 8. Delta transaction log — all operations recorded

Even though Type 1 keeps no dimension history, Delta Lake's own transaction log records every operation.  
You can always audit *when* a change happened, even if the old value is overwritten.

In [ ]:
spark.sql("""
    DESCRIBE HISTORY population_metrics.default.scd1_country_regions
""").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

## 9. (Optional) Cleanup

In [ ]:
# Uncomment to reset
# spark.sql("DROP TABLE IF EXISTS population_metrics.default.scd1_country_regions")
# print("Table dropped.")